# **Cybersecurity Spend Record Matching**

## **End-to-End Product Matching using Python and Fuzzy String Matching**

## **Technologies Used**

- Python
- Pandas
- RapidFuzz
- Regular Expressions

## **Project Overview**

This project develops an end-to-end product matching pipeline to identify standardized cybersecurity products from noisy procurement spend records.

The solution addresses common data quality challenges such as typographical errors, abbreviations, inconsistent naming conventions, and multiple product mentions by combining text normalization, exact string matching, and fuzzy matching techniques.

Each spend record is matched with the most likely product(s) from a product repository while providing confidence scores and the matching method used for transparency.

## **Business Problem**

Procurement data is often inconsistent and unstructured, making it difficult to accurately identify purchased cybersecurity products. Product names may contain spelling mistakes, abbreviations, vendor-specific terminology, or multiple products within a single record.

The objective of this project is to accurately map each spend record to one or more products from a standardized repository while maintaining an explainable and reproducible matching process.

## **Project Workflow**

1. Import Required Libraries
2. Load and Combine Datasets
3. Text Normalization
4. Repository Preparation
5. Product Matching
6. Confidence Scoring
7. Export Matched Records

**1.Environment & Libraries**

A small, standard stack is used: pandas for data wrangling, re for lightweight cleaning, and rapidfuzz for fuzzy matching. Heavy NLP or embedding methods are intentionally avoided to keep the solution transparent and reproducible.

In [ ]:
# Install RapidFuzz for fuzzy string matching
!pip install rapidfuzz

# Core libraries
import pandas as pd
import re
from rapidfuzz import fuzz, process

**2.Mount Google Drive**

Google Drive is mounted so inputs can be read and outputs can be saved to a stable location. Standardized paths under /content/drive/MyDrive/ help avoid file loss when the runtime resets.

In [ ]:
# Connect google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**3.Paths + Load + Combine (My Drive)**

Paths are defined and quickly checked (OK/MISSING). The product repository and both spend files are loaded. A dataset_source flag is added (easy/hard) and the two spend datasets are concatenated into a single table. Printed shapes provide a quick sanity check.

In [ ]:
import os, pandas as pd

PRODUCT_REPO_PATH = "/content/drive/MyDrive/product_repository.csv"
SPEND_EASY_PATH   = "/content/drive/MyDrive/cybersecurity_spend_records.csv"
SPEND_HARD_PATH   = "/content/drive/MyDrive/cybersecurity_spend_records_hard.csv"
OUTPUT_PATH       = "/content/drive/MyDrive/matched_spend_records.csv"

# quick path check
for p in [PRODUCT_REPO_PATH, SPEND_EASY_PATH, SPEND_HARD_PATH]:
    print(("OK  " if os.path.exists(p) else "MISS"), "—", p)

# load
repo = pd.read_csv(PRODUCT_REPO_PATH)
spend_easy = pd.read_csv(SPEND_EASY_PATH)
spend_hard = pd.read_csv(SPEND_HARD_PATH)

# tag + combine
spend_easy["dataset_source"] = "easy"
spend_hard["dataset_source"] = "hard"
spend = pd.concat([spend_easy, spend_hard], ignore_index=True)

print("Repository shape:", repo.shape)
print("Spend shape:", spend.shape)

OK   — /content/drive/MyDrive/product_repository.csv
OK   — /content/drive/MyDrive/cybersecurity_spend_records.csv
OK   — /content/drive/MyDrive/cybersecurity_spend_records_hard.csv
Repository shape: (119, 3)
Spend shape: (300, 8)


**4.Text normalization (make strings comparable)**

Text is standardized to make strings comparable: lowercase is applied, common symbol swaps are handled ($ to s, 0 to o, @ to a), punctuation and extra spaces are removed, and filler words (e.g., “license”, “support”) are dropped. This improves both exact and fuzzy matching.

In [ ]:
# Clean text so "Micro$oft", "microsoft!!", "MICROSOFT" all become comparable

STOPWORDS = {"license","licenses","licence","licences","renewal","support","subscription",
             "subs","annual","year","via","and","for","the","of","qty","ea","each",
             "service","services"}

LEET_MAP = str.maketrans({"$":"s","0":"o","@":"a","1":"i","5":"s","3":"e"})
PUNCT_RE = re.compile(r"[^\w\s]+")
SPACE_RE = re.compile(r"\s+")

def normalize(text):
    if pd.isna(text):
        return ""
    t = str(text).lower().translate(LEET_MAP)
    t = PUNCT_RE.sub(" ", t)
    t = SPACE_RE.sub(" ", t).strip()
    tokens = [w for w in t.split() if w not in STOPWORDS]
    return " ".join(tokens)

**5.Prepare repository**

Required columns are ensured. A simple sequential product_id is generated if absent, and vendor_name is defaulted to blank when missing. Normalized helper columns for product and vendor names are created for consistent, faster matching.

In [ ]:
# Ensure required columns exist
if "product_id" not in repo.columns:
    repo["product_id"] = range(1, len(repo) + 1)

if "vendor_name" not in repo.columns:
    repo["vendor_name"] = ""

# Minimal check for product_name
if "product_name" not in repo.columns:
    raise ValueError("Repo must have a 'product_name' column.")

# Normalized helper columns used for matching
repo["product_name_norm"] = repo["product_name"].map(normalize)
repo["vendor_name_norm"]  = repo["vendor_name"].map(normalize)

repo.head()

,id,product_name,vendor_name,product_id,product_name_norm,vendor_name_norm
0,PROD001,Defender for Enpoint,Microsoft,1,defender enpoint,microsoft
1,PROD002,CrowdStrike Falcon Enterprise,CrowdStrike,2,crowdstrike falcon enterprise,crowdstrike
2,PROD003,Cisco Firepower 2000 Series,Cisco,3,cisco firepower 2ooo series,cisco
3,PROD004,Wiz Cloud,Wiz,4,wiz cloud,wiz
4,PROD005,Wiz Defend,Wiz,5,wiz defend,wiz


**6.Matching logic (exact → fuzzy, simple & explainable)**

For each spend row, item and description are combined and normalized. A substring (exact) match is attempted first and given the highest confidence (100). If none is found, a fuzzy partial-ratio score is used with a practical threshold (85). The score and the method used are recorded for transparency.

In [ ]:
def match_record(item_text, description_text, repo, threshold=85):
    """
    Match one spend record to repository products.
    - Normalize item + description
    - Try exact/substring first
    - If nothing exact, try fuzzy partial ratio
    Returns a list of tuples: (product_id, product_name, vendor_name, score, method)
    """
    combined_text = normalize(f"{item_text or ''} {description_text or ''}")
    matches = []

    # Exact / substring match
    for _, r in repo.iterrows():
        if r["product_name_norm"] and r["product_name_norm"] in combined_text:
            matches.append((r["product_id"], r["product_name"], r["vendor_name"], 100, "exact"))

    # Fuzzy fallback if no exact
    if not matches:
        for _, r in repo.iterrows():
            score = fuzz.partial_ratio(combined_text, r["product_name_norm"])
            if score >= threshold:
                matches.append((r["product_id"], r["product_name"], r["vendor_name"], int(score), "fuzzy"))

    return matches


**7.Apply matcher to all rows (build final columns)**

The matching function is applied across all spend records. Multiple product hits per row are retained as lists. The best score becomes the row-level confidence; rows without a reliable hit are marked with confidence = 0 and method = "no_match".

In [ ]:
# Apply matcher to all rows (build final columns)
results = []

for _, row in spend.iterrows():
    # Run matcher on item + description for this row
    rec_matches = match_record(row.get("item",""), row.get("description",""), repo)

    if rec_matches:
      # Unpack per-product fields from (pid, name, vendor, score, method)
        product_ids   = [m[0] for m in rec_matches]
        product_names = [m[1] for m in rec_matches]
        vendor_names  = [m[2] for m in rec_matches]
        confidences   = [m[3] for m in rec_matches]
        methods       = [m[4] for m in rec_matches]

        best_conf   = max(confidences) if confidences else 0
        method_used = ",".join(set(methods))
    else:
       # No reliable match found
        product_ids, product_names, vendor_names = [], [], []
        best_conf, method_used = 0, "no_match"

    # Merge original fields with our new columns
    results.append({
          **row.to_dict(),
        "product_id": product_ids,
        "product_name": product_names,
        "vendor_name": vendor_names,
        "confidence": best_conf,
        "method": method_used
    })

# Build the final DataFrame and preview
matched_df = pd.DataFrame(results)
matched_df.head()


,id,supplier,item,description,cost,status,date,dataset_source,product_id,product_name,vendor_name,confidence,method
0,REQ87397,CrowdStrike Inc,Tanium + Microsft 365 E5 + Zscaler Internet Ac...,"As part of our cloud security strategy, the ac...",255150.87,DRAFT,2024-02-12,easy,[7],[Zscaler Internet Access],[Zscaler],100,exact
1,REQ54118,CC,Tanium Endpoint + Microsoft E5,Procurement of Tanium is being prioritised und...,182694.79,CANCELLED,2024-06-25,easy,[],[],[],0,no_match
2,REQ35203,Microsoft Corp,Z$caler IA + Microsft 365 E5,In response to recent threat intelligence repo...,387668.83,DRAFT,2024-04-21,easy,[],[],[],0,no_match
3,REQ44993,CrowdStrike Inc,Z$caler IA + Wiz Defend,Procurement of Z$caler IA is being prioritised...,269399.24,PENDING,2023-12-01,easy,[5],[Wiz Defend],[Wiz],100,exact
4,REQ45093,CC,Wiz Cl0ud + CS Falcon,In response to recent threat intelligence repo...,110250.14,ORDERED,2025-03-21,easy,[4],[Wiz Cloud],[Wiz],100,exact


**8.Save deliverable (tidy lists to CSV)**

List columns are flattened to a clean A | B | C format for readability in Excel. The required deliverable is written as matched_spend_records.csv in Google Drive.

In [ ]:
# Convert list columns to a clean "A | B | C" string so Excel looks nice and save
def tidy_list(x):
    if isinstance(x, list):
        # keep unique values, preserve order
        seen = []
        for v in x:
            if v not in seen:
                seen.append(str(v))
        return " | ".join(seen)
    return x

for col in ["product_id", "product_name", "vendor_name"]:
    matched_df[col] = matched_df[col].apply(tidy_list)

matched_df.to_csv(OUTPUT_PATH, index=False)
print("Saved tidy CSV to:", OUTPUT_PATH)

Saved tidy CSV to: /content/drive/MyDrive/matched_spend_records.csv


**9.Download CSV (Colab)**

The final CSV is made available for download directly from the notebook to facilitate quick review.

In [ ]:
# Download the file to your laptop from Colab
from google.colab import files
files.download("/content/drive/MyDrive/matched_spend_records.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>